# Campaign Investigator: run the agent

Runs the investigator with two open models on Colab's free GPU (Runtime > Change runtime type > T4 GPU). No API key needed. Roughly 30-60 minutes per model for the 12 cases.

In [ ]:
!git clone -q https://github.com/Zubair-DLCS/campaign-investigator.git
%cd campaign-investigator
!pip -q install -r requirements.txt

## 1. Run the non-LLM baselines

The 12 cases ship with the repository (`data/cases/`). This step downloads the Beyond Copy-Paste benchmark and its word vectors, which the detectors use (a few minutes).

In [ ]:
!python src/baselines.py

## 2. Start a local model server (Ollama) with a long context window

In [ ]:
!apt-get -qq install -y zstd > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null
import os, subprocess, time
env = dict(os.environ, OLLAMA_CONTEXT_LENGTH='16384')
subprocess.Popen(['ollama', 'serve'], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)

In [ ]:
MODELS = ['qwen2.5:7b', 'llama3.1:8b']   # any chat models from ollama.com/library
for m in MODELS:
    !ollama pull {m}

## 3. Run the investigator on every case

Resumable: if Colab disconnects, run this cell again and cached replies are reused.

In [ ]:
for m in MODELS:
    !python src/run.py --provider ollama --model {m}

## 4. Results and figures

In [ ]:
!python src/evaluate.py
!python src/figures.py
from IPython.display import Image, display
for f in ['fig1_overview.png', 'fig2_by_kind.png']:
    display(Image(f'figures/{f}'))

## 5. Download the results

In [ ]:
!zip -qr campaign_investigator_results.zip results figures -x 'results/cache_*'
from google.colab import files
files.download('campaign_investigator_results.zip')

## Optional: a hosted model

Any OpenAI-compatible provider works. Set `LLM_RPM` to the provider's requests-per-minute limit.

In [ ]:
import os
# os.environ['LLM_API_KEY'] = '...'      # never commit a key
# os.environ['LLM_RPM'] = '25'
# !python src/run.py --provider groq --model <model name from the provider's list>